# Visualisation & Debugging

Covers:
- Plotting the runtime execution graph (per-instance view)
- Plotting the static class dependency graph
- Inspecting loader parameters and hashes
- Cleaning stale cache entries

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import json
import numpy as np

from pygeodata import Data, SpatialSpec, setconfig
from pygeodata.processors.reprojection import Reprojector
from pygeodata.processors.rasterizer import Rasterizer
from pygeodata.visualisations import plot_compact_execution_graph, plot_class_dependency_graph
from pygeodata.cache import clean_cache
from pyproj import CRS
from affine import Affine

setconfig(path_data_processed=Path("./data/processed"))

spec = SpatialSpec(
    crs=CRS.from_epsg(4326),
    transform=Affine(0.5, 0, -180, 0, -0.5, 90),
    shape=(360, 720),
)

## Define a sample pipeline

In [ ]:
@dataclass
class RawDEMLoader(Data):
    src: str = "data/raw/dem.tif"

    @property
    def processor(self):
        return Reprojector(srcpath=self.src)


@dataclass
class SlopeMaskLoader(Data):
    src: Path = Path("data/raw/slope_mask.gpkg")

    @property
    def processor(self):
        return Rasterizer(srcpath=self.src, values=1, dtype=np.uint8, fill_value=0)


@dataclass
class MaskedDEMLoader(Data):
    dem: RawDEMLoader = None
    mask: SlopeMaskLoader = None

    def __post_init__(self):
        self.dem = self.dem or RawDEMLoader()
        self.mask = self.mask or SlopeMaskLoader()

    def process(self, spec):
        from pygeodata import load
        dem = load(self.dem, spec)
        mask = load(self.mask, spec)
        dem.where(mask == 1).rio.to_raster(self.get_processed_path(spec))


root = MaskedDEMLoader()

## 1. Plot the runtime execution graph

`plot_compact_execution_graph` renders the *runtime* dependency graph:
all `Data` instances wired via their parameters.
Nodes show class name, inherited classes, and parameter values.
Edges are labelled with the parameter name connecting two loaders.

Requires: `pip install graphviz` + system graphviz package.

In [ ]:
dot = plot_compact_execution_graph(
    loader=root,
    view=False,            # set True to open in your image viewer
    outpath="./data/processed/execution_graph",
    show_params=True,
    show_inheritance=True,
    show_calls=True,
)

# Display inline in Jupyter
from IPython.display import Image
Image("./data/processed/execution_graph.png")

## 2. Plot the static class dependency graph

`plot_class_dependency_graph` shows *class-level* relationships
(inheritance = solid arrow, method calls = dashed arrow)
rather than runtime instances.

In [ ]:
dot_cls = plot_class_dependency_graph(
    loader=MaskedDEMLoader,
    path=Path("./data/processed/source/MaskedDEMLoader/class_graph"),
    view=False,
)

Image("./data/processed/source/MaskedDEMLoader/class_graph.png")

## 3. Inspect parameters and hashes

In [ ]:
print("=== Parameters ===")
for k, v in root.get_params().items():
    print(f"  {k}: {v!r}")

print()
print("State hash (code + params): ", root.get_state_hash())
print("Source hash (code AST):     ", root.get_source_hash())
print("Hierarchy hash (full DAG):  ", root.get_source_hierarchy_hash())
print()
print("Output path:    ", root.get_processed_path(spec))
print("Is processed:   ", root.is_processed(spec))
print("Cache valid:    ", root.is_cache_valid(spec))

## 4. Inspect the full dependency tree

`get_dependency_tree()` returns a nested dict of class-level
dependencies (call and inheritance), with their AST hashes.
This is what the caching system uses to detect stale outputs.

In [ ]:
tree = MaskedDEMLoader.get_dependency_tree()
print(json.dumps(tree, indent=2))

## 5. Clean stale cache entries

`clean_cache` removes outputs whose source hierarchy hash no longer
matches the on-disk hash file (i.e., the code has changed).

In [ ]:
# Dry run — prints what would be deleted
clean_cache(loader=MaskedDEMLoader, dry_run=True)

# Remove stale outputs for a specific loader class:
# clean_cache(loader=MaskedDEMLoader, dry_run=False)

# Remove ALL stale outputs in the processed directory:
# clean_cache(dry_run=True)

## 6. Save the source registry

`initialize_source_registry()` writes the current class hash and
source code to disk. Called automatically by `process()`, but can
be triggered manually for inspection.

In [ ]:
MaskedDEMLoader.initialize_source_registry()
print("Registry path:", MaskedDEMLoader.get_source_registry_path())
print("Registry valid:", MaskedDEMLoader.is_source_registry_valid())